# Domain Adaptation: MMD Loss

Domain adaptation using Source (IN VITRO) + Target (PKSMART)
- align the source and target distributions with an MMD loss
- Shared encoder + task head
- AdamW optimizer

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

ROOT = r'..'
DATA_DIR = os.path.join(ROOT, 'data', 'cv_datasets')
FEAT_DIR = os.path.join(ROOT, 'features')
RES_DIR = os.path.join(ROOT, 'results', 'mmd_adaptation')
os.makedirs(RES_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROPERTIES = ['clearance', 'half_life', 'fu']
N_FOLDS = 5

print("="*100)
print("Domain Adaptation: MMD Loss")
print("="*100)
print(f"Device: {DEVICE}\n")

# Shared training utilities: inner train/val split + val-based early stopping (test used only for final evaluation)
sys.path.insert(0, os.path.join(ROOT, '01_model'))
from train_utils import split_train_val, fit_early_stopping, target_smiles


Domain Adaptation: MMD Loss
Device: cpu



In [2]:
print("Loading Graphormer embeddings...\n")

emb_path = os.path.join(FEAT_DIR, 'graphormer_cv_embeddings.npz')
emb_data = np.load(emb_path, allow_pickle=True)
emb_keys = emb_data['keys']
emb_X = emb_data['X']

smiles_to_emb = {smi: emb_X[i] for i, smi in enumerate(emb_keys)}

print(f"✓ Loaded {len(smiles_to_emb)} embeddings")
print(f"  Dim: {emb_X.shape[1]}\n")

Loading Graphormer embeddings...

✓ Loaded 58304 embeddings
  Dim: 768



In [3]:
def gaussian_kernel(x, y, sigma=1.0):
    x = x.unsqueeze(1)
    y = y.unsqueeze(0)
    distances = torch.sum((x - y) ** 2, dim=2)
    return torch.exp(-distances / (2 * sigma ** 2))

def compute_mmd(source_features, target_features, sigma=1.0):
    K_ss = gaussian_kernel(source_features, source_features, sigma)
    K_tt = gaussian_kernel(target_features, target_features, sigma)
    K_st = gaussian_kernel(source_features, target_features, sigma)
    
    n_s = source_features.shape[0]
    n_t = target_features.shape[0]
    
    mmd_loss = (1 / (n_s ** 2)) * K_ss.sum() + (1 / (n_t ** 2)) * K_tt.sum() - 2 * (1 / (n_s * n_t)) * K_st.sum()
    return torch.sqrt(torch.clamp(mmd_loss, min=1e-8))

print("✓ MMD loss defined")

✓ MMD loss defined


In [4]:
class SharedEncoder(nn.Module):
    def __init__(self, d_in=768, d_hidden=256, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.d_out = d_hidden // 2

    def forward(self, x):
        return self.encoder(x)

class TaskHead(nn.Module):
    def __init__(self, d_in, d_hidden=128):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, 1),
        )

    def forward(self, x):
        return self.head(x).squeeze(-1)

class DomainAdaptationModel(nn.Module):
    def __init__(self, d_in=768):
        super().__init__()
        self.encoder = SharedEncoder(d_in)
        self.task_head = TaskHead(self.encoder.d_out)

    def forward(self, x):
        features = self.encoder(x)
        output = self.task_head(features)
        return output, features

print("✓ Model defined")

✓ Model defined


In [5]:
def get_smiles_col(df):
    if 'smiles' in df.columns:
        return 'smiles'
    elif 'Drug' in df.columns:
        return 'Drug'
    else:
        return None

def get_value_col(df):
    if 'value' in df.columns:
        return 'value'
    elif 'Y' in df.columns:
        return 'Y'
    else:
        return None

def load_source(prop, fold_idx):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_source.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path)
    smi_col = get_smiles_col(df)
    val_col = get_value_col(df)
    
    if smi_col is None or val_col is None:
        return None, None
    
    smiles = df[smi_col].astype(str).tolist()
    
    X_list = []
    y_list = []
    for smi, y in zip(smiles, df[val_col].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    return X, y

def load_target(prop, fold_idx, split):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_target_{split}.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path)
    smi_col = get_smiles_col(df)
    val_col = get_value_col(df)
    
    if smi_col is None or val_col is None:
        return None, None
    
    smiles = df[smi_col].astype(str).tolist()
    
    X_list = []
    y_list = []
    for smi, y in zip(smiles, df[val_col].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    return X, y

print("✓ Data loaders defined")


✓ Data loaders defined


In [6]:
def train_epoch_mmd(model, X_src, y_src, X_tgt, y_tgt, optimizer, criterion, mmd_w=0.1, device='cpu'):
    model.train()
    
    # Ensure data is numpy array
    X_src = np.asarray(X_src, dtype='float32')
    y_src = np.asarray(y_src, dtype='float32')
    X_tgt = np.asarray(X_tgt, dtype='float32')
    y_tgt = np.asarray(y_tgt, dtype='float32')
    
    # Epochs count target batches; the source is randomly sampled from the whole set each batch (independent of source size)
    n = len(X_tgt)
    perm = np.random.permutation(n)
    
    task_loss_sum = 0.0
    mmd_loss_sum = 0.0
    n_batches = 0

    for i in range(0, n, 32):
        idx = perm[i:i + 32]
        idx_s = np.random.randint(0, len(X_src), len(idx))
        
        x_src = torch.tensor(X_src[idx_s], device=device, dtype=torch.float32)
        y_src_batch = torch.tensor(y_src[idx_s], device=device, dtype=torch.float32)
        x_tgt = torch.tensor(X_tgt[idx], device=device, dtype=torch.float32)
        y_tgt_batch = torch.tensor(y_tgt[idx], device=device, dtype=torch.float32)
        
        optimizer.zero_grad()
        
        y_pred_tgt, feat_tgt = model(x_tgt)
        task_loss = criterion(y_pred_tgt, y_tgt_batch)
        
        _, feat_src = model(x_src)
        mmd_loss = compute_mmd(feat_src, feat_tgt, sigma=1.0)
        
        total_loss = task_loss + mmd_w * mmd_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        task_loss_sum += task_loss.item()
        mmd_loss_sum += mmd_loss.item()
        n_batches += 1

    return task_loss_sum / max(n_batches, 1), mmd_loss_sum / max(n_batches, 1)

@torch.no_grad()
def evaluate(model, X, y, device):
    model.eval()
    X = np.asarray(X, dtype='float32')
    y = np.asarray(y, dtype='float32')
    
    x = torch.tensor(X, device=device, dtype=torch.float32)
    y_pred, _ = model(x)
    y_pred = y_pred.cpu().numpy()
    
    r = np.corrcoef(y, y_pred)[0, 1]
    rho, _ = spearmanr(y, y_pred)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(np.mean((y - y_pred) ** 2))
    mae = np.mean(np.abs(y - y_pred))
    mbe = np.mean(y_pred - y)
    
    return {'r': r, 'rho': rho, 'r2': r2, 'rmse': rmse, 'mae': mae, 'mbe': mbe}

print("✓ Training functions defined")


✓ Training functions defined


In [7]:
print("\n" + "="*100)
print("DOMAIN ADAPTATION: MMD Loss")
print("="*100 + "\n")

MMD_W = 0.1
results_dict = {}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    fold_results = []
    
    for fold_idx in range(N_FOLDS):
        print(f"\n  Fold {fold_idx}:")
        
        X_src, y_src = load_source(prop, fold_idx)
        X_tgt_train, y_tgt_train = load_target(prop, fold_idx, 'train')
        X_tgt_test, y_tgt_test = load_target(prop, fold_idx, 'test')
        
        if X_src is None or len(X_src) == 0:
            print(f"    No data")
            continue
        
        # The source is not truncated; everything is used (randomly sampled each batch during training)
        X_src = np.asarray(X_src, dtype='float32')
        y_src = np.asarray(y_src, dtype='float32')
        # target train → scaffold-based inner train/val (early stopping and λ selection on val, test used only for final evaluation)
        X_tgt_train, y_tgt_train, X_tgt_val, y_tgt_val = split_train_val(
            np.asarray(X_tgt_train, dtype='float32'), np.asarray(y_tgt_train, dtype='float32'), fold_idx,
                smiles=target_smiles(DATA_DIR, prop, fold_idx, 'train', smiles_to_emb))
        X_tgt_test = np.asarray(X_tgt_test, dtype='float32')
        y_tgt_test = np.asarray(y_tgt_test, dtype='float32')
        
        print(f"    Source: {X_src.shape}, Target: {X_tgt_train.shape} train, {X_tgt_test.shape} test")
        
        torch.manual_seed(fold_idx)
        np.random.seed(fold_idx)
        
        model = DomainAdaptationModel().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        criterion = nn.HuberLoss(delta=1.0)
        
        def run_epoch():
            train_epoch_mmd(model, X_src, y_src, X_tgt_train, y_tgt_train, optimizer, criterion, MMD_W, DEVICE)
        
        best_epoch, _ = fit_early_stopping(
            model, run_epoch, lambda m, X, y: evaluate(m, X, y, DEVICE), X_tgt_val, y_tgt_val)
        val_metrics = evaluate(model, X_tgt_val, y_tgt_val, DEVICE)
        metrics = evaluate(model, X_tgt_test, y_tgt_test, DEVICE)
        metrics.update({f'val_{k}': v for k, v in val_metrics.items()})
        metrics['best_epoch'] = best_epoch
        fold_results.append(metrics)
        print(f"    Result: rho={metrics['rho']:.4f}, r2={metrics['r2']:.4f}, rmse={metrics['rmse']:.4f}")
    
    if fold_results:
        results_dict[prop] = pd.DataFrame(fold_results)
        df = results_dict[prop]
        print(f"\n  {prop.upper()} SUMMARY")
        print(f"    rho: {df['rho'].mean():.4f} +/- {df['rho'].std():.4f}")
        print(f"    r2:  {df['r2'].mean():.4f} +/- {df['r2'].std():.4f}")
        print(f"    rmse: {df['rmse'].mean():.4f} +/- {df['rmse'].std():.4f}")

print(f"\n{'='*80}")
print(f"MMD Weight: {MMD_W}")
print("Compare with baseline (target-only) results")
print(f"{'='*80}\n")

for prop in PROPERTIES:
    if prop in results_dict:
        out_path = os.path.join(RES_DIR, f'{prop}.csv')
        results_dict[prop].to_csv(out_path, index=False)
        print(f"Saved: {out_path}")



DOMAIN ADAPTATION: MMD Loss


Property: CLEARANCE

  Fold 0:
    Source: (14386, 768), Target: (657, 768) train, (206, 768) test
    Result: rho=0.3003, r2=0.0386, rmse=0.9596

  Fold 1:
    Source: (14386, 768), Target: (657, 768) train, (206, 768) test
    Result: rho=0.3634, r2=0.1271, rmse=0.9250

  Fold 2:
    Source: (14386, 768), Target: (658, 768) train, (205, 768) test
    Result: rho=0.3827, r2=0.0989, rmse=0.8707

  Fold 3:
    Source: (14386, 768), Target: (658, 768) train, (205, 768) test
    Result: rho=0.4144, r2=0.1427, rmse=0.8683

  Fold 4:
    Source: (14386, 768), Target: (638, 768) train, (205, 768) test
    Result: rho=0.3830, r2=0.1060, rmse=1.1009

  CLEARANCE SUMMARY
    rho: 0.3688 +/- 0.0424
    r2:  0.1027 +/- 0.0398
    rmse: 0.9449 +/- 0.0953

Property: HALF_LIFE

  Fold 0:
    Source: (9202, 768), Target: (745, 768) train, (233, 768) test
    Result: rho=0.3357, r2=0.1004, rmse=0.9945

  Fold 1:
    Source: (9202, 768), Target: (745, 768) train, (233, 76

In [8]:
# Collect and compare results
print("\n" + "="*100)
print("MMD DOMAIN ADAPTATION RESULTS")
print("="*100 + "\n")

for prop in PROPERTIES:
    if prop in results_dict:
        df = results_dict[prop]
        print(f"{prop.upper()}:")
        print(f"  Spearman rho: {df['rho'].mean():.4f} ± {df['rho'].std():.4f}")
        print(f"  R2:           {df['r2'].mean():.4f} ± {df['r2'].std():.4f}")
        print(f"  RMSE:         {df['rmse'].mean():.4f} ± {df['rmse'].std():.4f}")
        print(f"  MAE:          {df['mae'].mean():.4f} ± {df['mae'].std():.4f}")
        print()

print("="*100)
print(f"MMD Weight: {MMD_W}")
print("Results saved to:", RES_DIR)
print("="*100)


MMD DOMAIN ADAPTATION RESULTS

CLEARANCE:
  Spearman rho: 0.3688 ± 0.0424
  R2:           0.1027 ± 0.0398
  RMSE:         0.9449 ± 0.0953
  MAE:          0.7100 ± 0.0485

HALF_LIFE:
  Spearman rho: 0.4035 ± 0.0575
  R2:           0.1015 ± 0.1061
  RMSE:         0.9394 ± 0.0690
  MAE:          0.7196 ± 0.0633

FU:
  Spearman rho: 0.5590 ± 0.0758
  R2:           0.2032 ± 0.1610
  RMSE:         0.8703 ± 0.0946
  MAE:          0.7022 ± 0.0708

MMD Weight: 0.1
Results saved to: ..\results\mmd_adaptation
